The implementation of this notebook is largely based on [the work by Vera Neplenbroek](https://github.com/Veranep/crosslingualdetoxdebias).

In [1]:
!pip install peft

  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (2.4 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.3 MB/s eta 0:00:00
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 5.3 MB/s eta 0:00:00
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-

In [1]:
from datasets import load_dataset
from peft import get_peft_model
from peft import prepare_model_for_kbit_training
from peft import LoraConfig
from peft import TaskType

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

In [2]:
HF_MODEL_NAME = "Qwen/Qwen2.5-0.5B"
NUM_TRAINING_SAMPLES = 50000 # up to 94966. Using an A100, training takes ~1.30 hours. On Colab's T4, training on 50k samples take approximately 4 hours.
NUM_PANDA_VAL_SAMPLES = 10551 # up to 10551. These are the eval samples used during training.
TRANSLATED_DATASET_DIR = "./data"
USE_RUSSIAN = True

In [3]:
from huggingface_hub import login
login()

In [4]:
def tokenize_panda(element):
    col = "perturbed_ru" if USE_RUSSIAN else "perturbed"
    outputs = tokenizer(
        element[col],
        truncation=True,
        max_length=512,
        padding="max_length",
        return_overflowing_tokens=True,
        return_length=True,
    )
    input_batch = []
    for input_ids in outputs["input_ids"]:
        input_batch.append(input_ids)
    return {"input_ids": input_batch}


In [5]:
print("Initialising Model")
peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        target_modules="all-linear",
        bias="none",
        r=64,
        lora_alpha=16,
        lora_dropout=0.1,
    )
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)
if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_NAME,
    device_map="auto",
    )

Initialising Model


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [6]:
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.config.pretraining_tp = 1
model.enable_input_require_grads()

if USE_RUSSIAN:
    from datasets import load_from_disk
    print("Loading translated Panda Dataset (Russian)")
    raw_datasets = load_from_disk(TRANSLATED_DATASET_DIR)
else:
    print("Loading Panda Dataset (English)")
    raw_datasets = load_dataset("facebook/panda")
valid_name = "validation"

print("Preparing Dataset")
tokenized_datasets = raw_datasets.map(
    tokenize_panda,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

Loading translated Panda Dataset (Russian)
Preparing Dataset


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10551 [00:00<?, ? examples/s]

In [7]:
training_args = TrainingArguments(
    output_dir=f"{HF_MODEL_NAME.split('/')[-1]}_lora_bias_sft_model",
    num_train_epochs=1,
    save_total_limit=5,
    eval_strategy="steps",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    warmup_steps=150,
    weight_decay=0.001,
    metric_for_best_model="eval_loss",
    fp16=True,
    remove_unused_columns=False,
    logging_steps=1000,
    eval_steps=1000,
    save_steps=1000,
    save_strategy="steps",
    learning_rate=3e-4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
    load_best_model_at_end=True,
    seed=42,
)
print("Getting PEFT Model ready")
model = get_peft_model(model, peft_config)

Getting PEFT Model ready


In [8]:
trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(NUM_TRAINING_SAMPLES)),
    eval_dataset=tokenized_datasets[valid_name].select(range(NUM_PANDA_VAL_SAMPLES)),
    data_collator=data_collator,
)

print("START TRAINING")
trainer.train()
debiased_model = model.merge_and_unload()

[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


START TRAINING


Step,Training Loss,Validation Loss
1000,2.137057,1.811595
2000,1.564809,1.393067
3000,1.259456,1.147156
4000,1.043790,0.992508
5000,0.944478,0.897168
6000,0.857902,0.847785
6250,0.857902,0.844866


In [9]:
#### ADD YOUR HUGGINGFACE REPO_ID HERE
debiased_model.push_to_hub("Qwen2.5-0.5B_russian_debias")
tokenizer.push_to_hub("Qwen2.5-0.5B_russian_debias")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/JDRIJKE/Qwen2.5-0.5B_russian_debias/commit/68e235e16d382fe7980b48cbbe4f1f40fa07846d', commit_message='Upload tokenizer', commit_description='', oid='68e235e16d382fe7980b48cbbe4f1f40fa07846d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/JDRIJKE/Qwen2.5-0.5B_russian_debias', endpoint='https://huggingface.co', repo_type='model', repo_id='JDRIJKE/Qwen2.5-0.5B_russian_debias'), pr_revision=None, pr_num=None)